# 주소로 위도·경도 가져오기
Geopy의 Nominatim으로 주소를 좌표로 변환하고, 변환한 좌표를 Folium 지도에 표시하는 실습입니다.

- Geopy 공식 문서: https://geopy.readthedocs.io/en/stable/
- 참고 자료: https://wikidocs.net/249024
- Nominatim은 무료 공개 서비스이므로 짧은 시간에 많은 요청을 보내지 않도록 요청 간격을 두어야 합니다.

In [1]:
# Geopy가 설치되지 않은 경우 터미널에서 아래 명령으로 프로젝트 의존성에 추가합니다.
# 이미 정상적으로 동작한다면 실행할 필요가 없습니다.
# uv add geopy

In [2]:
# Nominatim은 OpenStreetMap의 공개 지오코딩 서비스를 사용하는 클래스입니다.
# 지오코딩(geocoding)은 주소나 장소명을 위도·경도 좌표로 변환하는 작업입니다.
from geopy.geocoders import Nominatim
import time  # 여러 주소 요청 사이에 대기 시간을 둘 때 사용합니다.

# Nominatim 객체를 만듭니다. user_agent는 요청을 보내는 애플리케이션을 식별하는 필수 문자열입니다.
# 실제 서비스에서는 다른 사용자와 구분되는 고유하고 설명적인 값을 사용하는 것이 좋습니다.
geolocator = Nominatim(user_agent="happymaker1024")

# 좌표를 찾을 서울시청의 도로명 주소를 문자열로 저장합니다.
address = "서울특별시 중구 세종대로 110"

try:
    # geocode()가 외부 서비스에 주소를 요청하고 위치 객체를 반환합니다.
    location = geolocator.geocode(address)

    # 검색에 성공하면 위치 객체가 참(True), 실패하면 None이므로 먼저 존재 여부를 확인합니다.
    if location:
        # 위치 객체의 latitude와 longitude 속성에서 위도와 경도를 꺼냅니다.
        latitude = location.latitude
        longitude = location.longitude
        print(f"입력 주소: {address}")
        print(f"변환된 주소: {location.address}")
        print(f"위도: {latitude}")
        print(f"경도: {longitude}")
    else:
        # 주소가 모호하거나 검색 결과가 없으면 location이 None이 됩니다.
        print(f"'{address}' 주소를 찾을 수 없습니다.")

# 네트워크 문제 등 실행 중 발생할 수 있는 예외를 잡아 노트북이 중단되지 않게 합니다.
except Exception as e:
    print(f"지오코딩 중 오류 발생: {e}")

# 여러 주소를 처리할 때는 공개 서버에 부담을 주지 않도록 요청 사이에 대기 시간을 둡니다.
# time.sleep(1)  # 한 주소만 검색하는 현재 예제에서는 실행하지 않습니다.

입력 주소: 서울특별시 중구 세종대로 110
변환된 주소: 서울특별시청, 110, 세종대로, 태평로1가, 명동, 중구, 서울특별시, 04524, 대한민국
위도: 37.5667893
경도: 126.9784204


In [3]:
# Folium은 위도·경도 좌표를 웹 기반의 대화형 지도에 표시하는 라이브러리입니다.
import folium

In [4]:
# 앞 셀에서 주소로 얻은 latitude와 longitude를 중심으로 Folium 지도 객체를 만듭니다.
# location은 [위도, 경도] 순서이며 zoom_start가 클수록 지도가 더 확대됩니다.
foliumMap=folium.Map(location=[latitude, longitude], zoom_start=16)

# Marker를 같은 좌표에 만들고 add_to()로 지도 객체에 추가합니다.
folium.Marker(location=[latitude, longitude]).add_to(foliumMap)
# Popup은 마커를 클릭했을 때 표시되는 정보창이며 max_width는 최대 너비입니다.
popup=folium.Popup('서울시청위치.', max_width=300)
folium.Marker(location=[latitude, longitude], popup=popup).add_to(foliumMap)

# 지도를 HTML로 저장할 폴더를 준비합니다. exist_ok=True이면 이미 있어도 오류가 나지 않습니다.
import os
os.makedirs('chart_datas', exist_ok=True)
# 브라우저에서 열 수 있는 대화형 지도 파일을 저장합니다.
foliumMap.save('chart_datas/folium_map0.html')
# 셀의 마지막 줄에 지도 객체를 두면 Jupyter에도 지도가 출력됩니다.
foliumMap

In [5]:
# 주소 검색 결과 대신 서울시청의 위도·경도를 숫자로 직접 입력하는 방식입니다.
foliumMap=folium.Map(location=[37.5662952, 126.9779451], zoom_start=16)

# 같은 고정 좌표에 기본 마커와 팝업 마커를 차례로 추가합니다.
folium.Marker(location=[37.5662952, 126.9779451]).add_to(foliumMap)
popup=folium.Popup('서울시청위치.', max_width=300)
folium.Marker(location=[37.5662952, 126.9779451], popup=popup).add_to(foliumMap)
# 앞 셀과 파일명이 같으므로 실행하면 기존 HTML 파일을 새 지도로 덮어씁니다.
foliumMap.save('chart_datas/folium_map0.html')
# 완성된 지도를 노트북에 출력합니다.
foliumMap

In [1]:
# 여러 주소를 연속으로 좌표로 변환하는 예제입니다. 필요한 클래스와 time을 불러옵니다.
from geopy.geocoders import Nominatim
import time

# Nominatim 요청에 필요한 지오코더 객체를 만듭니다. user_agent는 필수 식별 문자열입니다.
geolocator = Nominatim(user_agent="happymaker1024")

# 좌표로 변환할 주소들을 Python 리스트에 저장합니다.
addresses = ["서울특별시 중구", "서울특별시 강남구", "부산광역시 해운대구", "경기도 성남시 분당구"]

# for문이 리스트의 주소를 하나씩 addr 변수에 넣어 반복합니다.
for addr in addresses:
    location = geolocator.geocode(addr)  # 현재 주소를 위치 객체로 변환합니다.
    print(location.latitude, location.longitude)  # 위도와 경도를 출력합니다.
    time.sleep(1)  # 공개 서버 보호를 위해 다음 요청 전 1초간 대기합니다.


37.5636559 126.9975097
37.5177 127.0473
35.1629 129.1638
37.3825999 127.1188
